In [75]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from datetime import datetime
import contractions
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint
import re

In [76]:

train_df = pd.read_csv("./data/Corona_NLP_train.csv", encoding="latin1")
test_df = pd.read_csv("./data/Corona_NLP_test.csv", encoding="latin1")

train_df.head(10)

,UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment
0,3799,48751,London,16-03-2020,@MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...,Neutral
1,3800,48752,UK,16-03-2020,advice Talk to your neighbours family to excha...,Positive
2,3801,48753,Vagabonds,16-03-2020,Coronavirus Australia: Woolworths to give elde...,Positive
3,3802,48754,NaN,16-03-2020,My food stock is not the only one which is emp...,Positive
4,3803,48755,NaN,16-03-2020,"Me, ready to go at supermarket during the #COV...",Extremely Negative
5,3804,48756,"ÃT: 36.319708,-82.363649",16-03-2020,As news of the regionÂs first confirmed COVID...,Positive
6,3805,48757,"35.926541,-78.753267",16-03-2020,Cashier at grocery store was sharing his insig...,Positive
7,3806,48758,Austria,16-03-2020,Was at the supermarket today. Didn't buy toile...,Neutral
8,3807,48759,"Atlanta, GA USA",16-03-2020,Due to COVID-19 our retail store and classroom...,Positive
9,3808,48760,"BHAVNAGAR,GUJRAT",16-03-2020,"For corona prevention,we should stop to buy th...",Negative


# Data cleaning

In [77]:
def process_date(df: pd.DataFrame):
    parsed_dates = df["TweetAt"].apply(lambda date_str: datetime.strptime(date_str, "%d-%m-%Y"))
    
    df["day"] = parsed_dates.dt.day
    df["month"] = parsed_dates.dt.month
    df["year"] = parsed_dates.dt.year

    df.drop("TweetAt", axis=1, inplace=True)

def preprocess(tweet: str) -> str:
    text = tweet.encode('utf-8').decode('utf-8')
    text = text.lower()
    text = re.sub(r"\bhttps?:\/\/[^\s/$.?#].[^\s]*\b", "", text) # remove URLs
    text = re.sub(r"#\w+", "", text) # remove Hashtags
    text = re.sub(r"@\w+", "", text) # remove @ mentions
    text = contractions.fix(text)
    return text

train_df.drop(axis=0, columns=["Location", "UserName", "ScreenName"], inplace=True)
test_df.drop(axis=0, columns=["Location", "UserName", "ScreenName"], inplace=True)

train_df["OriginalTweet"] = train_df["OriginalTweet"].apply(preprocess)
test_df["OriginalTweet"] = test_df["OriginalTweet"].apply(preprocess)

process_date(train_df)
process_date(test_df)

train_df.head(10)

,OriginalTweet,Sentiment,day,month,year
0,and and,Neutral,16,3,2020
1,advice talk to your neighbours family to excha...,Positive,16,3,2020
2,coronavirus australia: woolworths to give elde...,Positive,16,3,2020
3,my food stock is not the only one which is emp...,Positive,16,3,2020
4,"me, ready to go at supermarket during the out...",Extremely Negative,16,3,2020
5,as news of the regionâs first confirmed covid...,Positive,16,3,2020
6,cashier at grocery store was sharing his insig...,Positive,16,3,2020
7,was at the supermarket today. did not buy toil...,Neutral,16,3,2020
8,due to covid-19 our retail store and classroom...,Positive,16,3,2020
9,"for corona prevention,we should stop to buy th...",Negative,16,3,2020


In [78]:

# Define features
text_col = "OriginalTweet"
num_cols = ["day", "month", "year"]

# Prepare data
X_train = train_df[[text_col] + num_cols]
y_train = train_df["Sentiment"]
X_test = test_df[[text_col] + num_cols]
y_test = test_df["Sentiment"]

# Preprocessing pipeline
preprocessor = ColumnTransformer([
    ("text", TfidfVectorizer(min_df=3, max_df=0.7), text_col),
    ("num", StandardScaler(), num_cols),
])

# Full pipeline
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42)),
])

# Define hyperparameter search space
param_dist = {
    "preprocessor__text__ngram_range": [(1,1), (1,2), (1,3)],
    "preprocessor__text__max_features": [5000, 10000, 20000],
    "preprocessor__text__min_df": [1, 2, 3, 5],
    "preprocessor__text__max_df": [0.6, 0.7, 0.8, 0.9],
    "classifier__C": uniform(0.01, 10),
}
# Randomized search
random_search = RandomizedSearchCV(
    model,
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2,
    random_state=42
)

# Fit and evaluate
random_search.fit(X_train, y_train)
print("Best params:", random_search.best_params_)
print("Best score:", random_search.best_score_)

# Use best model on test set
best_model = random_search.best_estimator_
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best params: {'classifier__C': np.float64(7.861759613930136), 'preprocessor__text__max_df': 0.8, 'preprocessor__text__max_features': 20000, 'preprocessor__text__min_df': 5, 'preprocessor__text__ngram_range': (1, 1)}
Best score: 0.5615812619967442
                    precision    recall  f1-score   support

Extremely Negative       0.62      0.68      0.65       592
Extremely Positive       0.66      0.66      0.66       599
          Negative       0.61      0.53      0.57      1041
           Neutral       0.67      0.79      0.72       619
          Positive       0.58      0.56      0.57       947

          accuracy                           0.62      3798
         macro avg       0.63      0.64      0.63      3798
      weighted avg       0.62      0.62      0.62      3798

